# WRO 2026 — 3-class detector (green / red / magenta)
Trains YOLO26 on the labelled red/green pillar set + the 2026-08-12 magenta lot-marker set.

**Before Run All:** Runtime → Change runtime type → **GPU**, then fill the two paths in the config cell.

Class order is LOCKED to the project lineage: **0=green, 1=red, 2=magenta**. Inverting it makes the robot pass every pillar on the wrong side.

In [ ]:
# ---------- CONFIG ----------
RG_DIR      = '/content/drive/MyDrive/CHANGE_ME/red_green_dataset'  # folder of the labelled red/green YOLO export (has train/valid dirs or images+labels)
MAGENTA_ZIP = '/content/drive/MyDrive/magenta_yolo_2026-08-12.zip'  # upload the zip from the laptop to Drive first
TINY_SPLITS = ''   # OPTIONAL: Drive folder holding val.txt from desktop wro_vision/tiny/splits (the clean group-wise split). Blank = in-notebook fallback split.
MODEL  = 'yolo26n.pt'   # n, not s: the in-system 0.3 fps collapse is suspected OOM; n cuts the memory suspect. Switch to yolo26s.pt only with a reason.
IMGSZ  = 320            # 224 starves distant pillars (min box ~13 px); 320 per the 07-26 trade analysis
EPOCHS = 80

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import zipfile, os, shutil
if os.path.exists('/content/ds'): shutil.rmtree('/content/ds')
os.makedirs('/content/ds/mag')
zipfile.ZipFile(MAGENTA_ZIP).extractall('/content/ds/mag')
print('magenta zip extracted')

In [ ]:
# ---------- MERGE + GROUP-WISE SPLIT (the shipped red/green split is LEAKY - never reuse it) ----------
import glob, os, shutil, random, re, collections
random.seed(0)
root = '/content/ds'
for s in ('train','val'):
    for k in ('images','labels'): os.makedirs(f'{root}/{s}/{k}', exist_ok=True)

def put(img, lbl, split, tag):
    b = os.path.basename(img)
    shutil.copy2(img, f'{root}/{split}/images/{tag}{b}')
    dst = f'{root}/{split}/labels/{tag}{os.path.splitext(b)[0]}.txt'
    if lbl and os.path.exists(lbl): shutil.copy2(lbl, dst)
    else: open(dst,'w').close()

# magenta: pre-split in the zip (vidA=train incl. augs, vidB=val) - copy through
for split in ('train','val'):
    for img in sorted(glob.glob(f'{root}/mag/magenta/{split}/images/*.jpg')):
        put(img, img.replace('/images/','/labels/')[:-4]+'.txt', split, 'mag_')

# red/green: gather every image, derive its label, DEDUPE stems, then split by GROUP
imgs = sorted(glob.glob(RG_DIR + '/**/*.jpg', recursive=True)) + sorted(glob.glob(RG_DIR + '/**/*.png', recursive=True))
pairs, missing = {}, 0
for p in imgs:
    stem = os.path.splitext(os.path.basename(p))[0]
    if stem in pairs: continue                      # 25 duplicate stems exist in this export
    l = p.replace('/images/','/labels/')
    l = os.path.splitext(l)[0] + '.txt'
    if not os.path.exists(l):
        l2 = os.path.splitext(p)[0] + '.txt'
        l = l2 if os.path.exists(l2) else None
    if l is None: missing += 1
    pairs[stem] = (p, l)
print(f'red/green unique stems: {len(pairs)}, missing labels: {missing}')
assert len(pairs) > 0, 'RG_DIR wrong - no images found'
assert missing <= 0.05*len(pairs), 'Too many red/green images have NO label file - they would train as background and teach the model to ignore pillars. Fix RG_DIR / labels first.'

val_stems = None
if TINY_SPLITS and os.path.exists(os.path.join(TINY_SPLITS,'val.txt')):
    val_stems = {os.path.splitext(os.path.basename(x))[0] for x in open(os.path.join(TINY_SPLITS,'val.txt')).read().split() if x.strip()}
    print(f'USING DESKTOP MANIFEST SPLIT: {len(val_stems)} val stems')
if val_stems:
    for stem,(p,l) in pairs.items(): put(p, l, 'val' if stem in val_stems else 'train', 'rg_')
else:
    print('WARNING: fallback in-notebook group split (weaker than the desktop manifests - copy wro_vision/tiny/splits/val.txt to Drive for the clean one)')
    def gkey(stem):
        nums = re.findall(r'\d+', stem)
        return (re.sub(r'\d+','', stem)[:24], int(nums[-1])//5 if nums else 0)
    groups = collections.defaultdict(list)
    for stem,(p,l) in pairs.items(): groups[gkey(stem)].append((p,l))
    keys = sorted(groups); random.shuffle(keys)
    vset = set(keys[:max(1, int(0.2*len(keys)))])
    print(f'groups: {len(keys)}, val groups: {len(vset)}')
    for k, items in groups.items():
        for p,l in items: put(p, l, 'val' if k in vset else 'train', 'rg_')

tr = len(glob.glob(root+'/train/images/*')); va = len(glob.glob(root+'/val/images/*'))
print(f'TOTAL train {tr} / val {va}')

In [ ]:
# ---------- CLASS-ID SANITY + data.yaml ----------
import glob
def ids(pattern):
    s = set()
    for f in glob.glob(pattern):
        for ln in open(f):
            if ln.strip(): s.add(int(ln.split()[0]))
    return s
rg, mg = ids('/content/ds/*/labels/rg_*.txt'), ids('/content/ds/*/labels/mag_*.txt')
print('red/green label ids:', rg, '| magenta label ids:', mg)
assert rg <= {0,1}, f'red/green labels carry ids {rg} - expected only 0(green)/1(red). STOP: class-order landmine.'
assert mg <= {2},  f'magenta labels carry ids {mg} - expected only 2. STOP.'
with open('/content/ds/data.yaml','w') as f:
    f.write('path: /content/ds\ntrain: train/images\nval: val/images\nnames:\n  0: green\n  1: red\n  2: magenta\n')
print('data.yaml written - CLASS ORDER LOCKED: 0=green 1=red 2=magenta')

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO
model = YOLO(MODEL)
model.train(data='/content/ds/data.yaml', imgsz=IMGSZ, epochs=EPOCHS,
            patience=20, batch=-1, seed=0,
            hsv_h=0.0)   # HUE JITTER OFF - hue IS the class label (red vs magenta live on it)

In [ ]:
# ---------- VALIDATE (per-class) + EXPORT ----------
from ultralytics import YOLO
best = YOLO('runs/detect/train/weights/best.pt')
m = best.val(data='/content/ds/data.yaml', imgsz=IMGSZ)
for i, name in enumerate(['green','red','magenta']):
    try: print(f'{name:8s} mAP50-95 = {m.box.maps[i]:.3f}')
    except Exception: pass
best.export(format='onnx', imgsz=IMGSZ, dynamic=False)  # FIXED shape - dynamic blocks NCNN/int8 optimisation
best.export(format='ncnn', imgsz=IMGSZ)
print('exports done')

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/wro_3class_out', 'zip', 'runs/detect/train/weights')
files.download('/content/wro_3class_out.zip')
print('DONE. On-Pi acceptance before this replaces anything: benchncnn latency AND the concurrent-load test - the last YOLO did 30 fps isolated and 0.3 fps in-system.')